In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install tb-nightly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 74.5 MB/s eta 0:00:00


In [ ]:
!pip install basicsr

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.5/172.5 kB 5.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.4/299.4 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 12.5 MB/s eta 0:00:00
  Created wheel for basicsr: filename=basicsr-1.4.2-py3-none-any.whl size=214818 sha256=3b0e4dedc2959de8df8a8bd3666cd55c17dde98c0455002b261cb607a002a25e
  Stored in directory: /root/.cache/pip/wheels/9a/e3/e4/58f29bfabb622dd40b6d9839318ce5bf092062b81ca3aa19ea
Successfully built basicsr


In [ ]:
!pip install facexlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 6.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 5.0 MB/s eta 0:00:00
  Created wheel for filterpy: filename=filterpy-1.4.5-py3-none-any.whl size=110460 sha256=e75935f4727e5467f24d803fd4d35f9738bc6552e2dbbe497db4e5a9b2625b95
  Stored in directory: /root/.cache/pip/wheels/77/bf/4c/b0c3f4798a0166668752312a67118b27a3cd341e13ac0ae6ee
Successfully built filterpy


In [ ]:
!pip install gfpgan

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.2/52.2 kB 2.6 MB/s eta 0:00:00


In [ ]:
!pip install realesrgan

In [ ]:
!pip install opencv-python numpy

In [ ]:
!mkdir results

In [ ]:
!mkdir inputs

In [ ]:
import os

In [ ]:
import sys
import os
import torch
import cv2
import numpy as np
import glob
from torch.nn import functional as F_torch

In [ ]:
if not os.path.exists('HAT'):
    print("⬇️ Cloning HAT Repository (contains the missing model files)...")
    !git clone https://github.com/XPixelGroup/HAT.git

⬇️ Cloning HAT Repository (contains the missing model files)...
Cloning into 'HAT'...
remote: Enumerating objects: 419, done.
remote: Counting objects: 100% (242/242), done.
remote: Compressing objects: 100% (122/122), done.
remote: Total 419 (delta 197), reused 120 (delta 120), pack-reused 177 (from 2)
Receiving objects: 100% (419/419), 20.72 MiB | 15.45 MiB/s, done.
Resolving deltas: 100% (232/232), done.


In [ ]:
if os.path.abspath('HAT') not in sys.path:
    sys.path.insert(0, os.path.abspath('HAT'))

In [ ]:
%cd HAT

/content/HAT


In [ ]:
try:
    # TRY 1: The correct path for XPixelGroup/HAT repo
    from hat.archs.hat_arch import HAT
    from basicsr.utils import img2tensor, tensor2img
    print("✅ Success! Imported from 'hat.archs.hat_arch'")
except ImportError as e:
    print(f"⚠️ Import Error: {e}")
    print("Trying fallback import...")
    # TRY 2: Fallback if folder structure is slightly different
    try:
        from basicsr.archs.hat_arch import HAT
        from basicsr.utils import img2tensor, tensor2img
    except ImportError:
        raise ImportError("❌ Could not find 'HAT' model. Make sure the 'HAT' folder is in your files.")

✅ Success! Imported from 'hat.archs.hat_arch'


In [ ]:
try:
    from torchvision.transforms import functional as F
    sys.modules["torchvision.transforms.functional_tensor"] = F
except ImportError:
    pass

In [ ]:
import os
import requests

# 1. Define the correct URLs (Using HuggingFace mirror for direct access)
# We use the 'resolve/main' format which forces a direct download
file_url = "https://huggingface.co/jaideepsingh/upscale_models/resolve/main/HAT/HAT-L_SRx4_ImageNet-pretrain.pth"
target_filename = "HAT_L_ImageNet-pretrain.pth"

# 2. Download Function with Progress Bar
def download_model(url, filename):
    print(f"⬇️ Downloading from reliable mirror:\n   {url}")
    try:
        response = requests.get(url, stream=True)
        response.raise_for_status()  # Check for 404/500 errors

        total_size = int(response.headers.get('content-length', 0))
        block_size = 1024 * 1024  # 1 MB
        wrote = 0

        with open(filename, 'wb') as f:
            for data in response.iter_content(block_size):
                wrote += len(data)
                f.write(data)
                if total_size > 0:
                    percent = (wrote / total_size) * 100
                    print(f"\r   Progress: {percent:.1f}% ({wrote // (1024*1024)} MB)", end="")

        print("\n✅ Download Complete!")
        return True
    except Exception as e:
        print(f"\n❌ Error downloading: {e}")
        return False

# 3. Run Download
# Clean up old partial files first
if os.path.exists(target_filename):
    os.remove(target_filename)

success = download_model(file_url, target_filename)

# 4. Verification
if success and os.path.exists(target_filename):
    size_mb = os.path.getsize(target_filename) / (1024 * 1024)
    print(f"🎉 SUCCESS! Model is ready.")
    print(f"📂 File: {target_filename}")
    print(f"📊 Size: {size_mb:.2f} MB")

    if size_mb < 100:
        print("⚠️ WARNING: File seems too small for HAT-L. It might be the Standard (S) version or corrupt.")
    else:
        print("✅ Size looks correct for HAT-L (Large).")
else:
    print("❌ Failed. Please check your internet connection.")

⬇️ Downloading from reliable mirror:
   https://huggingface.co/jaideepsingh/upscale_models/resolve/main/HAT/HAT-L_SRx4_ImageNet-pretrain.pth
   Progress: 100.0% (158 MB)
✅ Download Complete!
🎉 SUCCESS! Model is ready.
📂 File: HAT_L_ImageNet-pretrain.pth
📊 Size: 158.09 MB
✅ Size looks correct for HAT-L (Large).


In [ ]:
import os

file_path = 'HAT_L_ImageNet-pretrain.pth'

if os.path.exists(file_path):
    size_mb = os.path.getsize(file_path) / (1024 * 1024)
    print(f"✅ SUCCESS: Model found!")
    print(f"📂 File: {file_path}")
    print(f"📊 Size: {size_mb:.2f} MB (Should be around 350-400 MB)")
else:
    print("❌ ERROR: File not found. The download failed.")

✅ SUCCESS: Model found!
📂 File: HAT_L_ImageNet-pretrain.pth
📊 Size: 158.09 MB (Should be around 350-400 MB)


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [ ]:
base_dir = '/content'
repo_dir = os.path.join(base_dir, 'HAT')

if os.path.exists(repo_dir):
    os.chdir(repo_dir) # Change working directory to /content/HAT
    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)
    print(f"📂 Changed directory to: {os.getcwd()}")
else:
    print("❌ Error: HAT folder not found. Please ensure it is cloned.")

weights_path = 'HAT_L_ImageNet-pretrain.pth'
model = HAT(
    upscale=4,
    in_chans=3,
    img_size=64,
    window_size=16,
    compress_ratio=3,
    depths=[6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6],
    embed_dim=180,
    num_heads=[6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6],
    mlp_ratio=2,
    resi_connection='1conv',
    upsampler='pixelshuffle'
)

if os.path.exists(weights_path):
    print(f"⬇️ Loading weights from: {os.path.abspath(weights_path)}")

    # Load to GPU immediately
    checkpoint = torch.load(weights_path, map_location=device)

    # Use the 'params_ema' key verified in your previous check
    if 'params_ema' in checkpoint:
        model.load_state_dict(checkpoint['params_ema'])
    else:
        model.load_state_dict(checkpoint)

    model.eval()
    model = model.to(device)
    print("✅ SUCCESS: HAT-L Model Loaded and Ready on GPU!")

    # Return to main content folder for easier file management
    os.chdir(base_dir)
    print(f"🏠 Returned to: {os.getcwd()}")
else:
    print(f"❌ Error: Weights file not found at {os.path.abspath(weights_path)}")

🖥️ Using device: cuda
📂 Changed directory to: /content/HAT
⬇️ Loading weights from: /content/HAT/HAT_L_ImageNet-pretrain.pth
✅ SUCCESS: HAT-L Model Loaded and Ready on GPU!
🏠 Returned to: /content


In [ ]:
def img2tensor(img):
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.
    return torch.from_numpy(np.transpose(img, (2, 0, 1)))

def tensor2img(tensor):
    output = tensor.squeeze(0).detach().cpu().numpy()
    output = np.transpose(output, (1, 2, 0))
    output = cv2.cvtColor(output, cv2.COLOR_RGB2BGR)
    return (np.clip(output, 0, 1) * 255.0).round().astype(np.uint8)

def pad_image(img_tensor, window_size=16):
    """Simplified padding to avoid 'mode' keyword errors."""
    _, _, h, w = img_tensor.size()
    pad_h = (window_size - h % window_size) % window_size
    pad_w = (window_size - w % window_size) % window_size
    img_tensor = F.pad(img_tensor, (0, pad_w, 0, pad_h), 'reflect')
    return img_tensor, h, w

In [ ]:
input_files = sorted(glob.glob("/content/inputs/*"))
output_folder = "/content/results/path_a_hat"
os.makedirs(output_folder, exist_ok=True)

print(f"🚀 Retrying {len(input_files)} images on GPU...")

for img_path in input_files:
    filename = os.path.basename(img_path)
    try:
        img = cv2.imread(img_path)
        if img is None: continue

        img_t = img2tensor(img).unsqueeze(0).to(device)
        img_t, h_old, w_old = pad_image(img_t)

        with torch.no_grad():
            output_t = model(img_t)

        # HAT scales by 4x, so we crop the padded areas back out
        output_t = output_t[:, :, :h_old*4, :w_old*4]

        cv2.imwrite(os.path.join(output_folder, filename), tensor2img(output_t))
        print(f"   ✅ Processed: {filename}")

    except Exception as e:
        print(f"⚠️ Error on {filename}: {e}")

print(f"\n✨ Success! Path A images are saved in: {output_folder}")

🚀 Retrying 12 images on GPU...
   ✅ Processed: 00000.png
   ✅ Processed: 00001.png
   ✅ Processed: 00002.png
   ✅ Processed: 00003.png
   ✅ Processed: 00004.png
   ✅ Processed: 00005.png
   ✅ Processed: 00006.png
   ✅ Processed: 00007.png
   ✅ Processed: 00008.png
   ✅ Processed: 00009.png
   ✅ Processed: 00010.png
   ✅ Processed: 00011.png

✨ Success! Path A images are saved in: /content/results/path_a_hat


In [ ]:
%cd /content

/content


In [ ]:
if not os.path.exists('CodeFormer'):
    print("⬇️ Cloning CodeFormer...")
    !git clone https://github.com/sczhou/CodeFormer.git
    %cd CodeFormer

⬇️ Cloning CodeFormer...
Cloning into 'CodeFormer'...
remote: Enumerating objects: 620, done.
remote: Counting objects: 100% (303/303), done.
remote: Compressing objects: 100% (118/118), done.
remote: Total 620 (delta 212), reused 185 (delta 185), pack-reused 317 (from 2)
Receiving objects: 100% (620/620), 17.32 MiB | 15.38 MiB/s, done.
Resolving deltas: 100% (300/300), done.
/content/CodeFormer


In [ ]:
!pip uninstall basicsr -y

Found existing installation: basicsr 1.4.2
Uninstalling basicsr-1.4.2:
  Successfully uninstalled basicsr-1.4.2


In [ ]:
!pip install basicsr

In [ ]:
%cd CodeFormer

/content/CodeFormer


In [ ]:
!pip install -r requirements.txt

In [ ]:
%cd /content/CodeFormer

/content/CodeFormer


In [ ]:
!python basicsr/setup.py develop

/usr/local/lib/python3.12/dist-packages/setuptools/__init__.py:94: _DeprecatedInstaller: setuptools.installer and fetch_build_eggs are deprecated.
!!

        ********************************************************************************
        Requirements should be satisfied by a PEP 517 installer.
        If you are using pip, you can try `pip install --use-pep517`.
        ********************************************************************************

!!
  dist.fetch_build_eggs(dist.setup_requires)
running develop
/usr/local/lib/python3.12/dist-packages/setuptools/command/develop.py:41: EasyInstallDeprecationWarning: easy_install command is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` and ``easy_install``.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://github.com/pypa/setuptools/issues/917 for details.
        *****

In [ ]:
!pip uninstall basicsr -y

Found existing installation: basicsr 1.3.2
Uninstalling basicsr-1.3.2:
  Successfully uninstalled basicsr-1.3.2


In [ ]:
codeformer_path = '/content/CodeFormer'
if codeformer_path not in sys.path:
    sys.path.insert(0, codeformer_path)

In [ ]:
local_patch_file = f'{codeformer_path}/basicsr/data/degradations.py'
if os.path.exists(local_patch_file):
    print(f"🛠️ Patching local file: {local_patch_file}")
    !sed -i 's/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/g' {local_patch_file}

In [ ]:
local_version_file = f'{codeformer_path}/basicsr/version.py'
with open(local_version_file, 'w') as f:
    f.write("__gitsha__ = 'unknown'\n__version__ = '1.4.2'")
print("✅ Local version.py created.")

✅ Local version.py created.


In [ ]:
!pip install basicsr

  Using cached basicsr-1.4.2-py3-none-any.whl


In [ ]:
%cd /content/CodeFormer

/content/CodeFormer


In [ ]:
# Cell: Direct Model Download (Bypass Python Errors)
import os

# Create weights directories
!mkdir -p /content/CodeFormer/weights/CodeFormer
!mkdir -p /content/CodeFormer/weights/facelib

print("⬇️ Downloading CodeFormer model...")
!wget https://github.com/sczhou/CodeFormer/releases/download/v0.1.0/codeformer.pth -O /content/CodeFormer/weights/CodeFormer/codeformer.pth

print("⬇️ Downloading Detection models...")
!wget https://github.com/sczhou/CodeFormer/releases/download/v0.1.0/detection_Resnet50_Final.pth -O /content/CodeFormer/weights/facelib/detection_Resnet50_Final.pth
!wget https://github.com/sczhou/CodeFormer/releases/download/v0.1.0/parsing_parsenet.pth -O /content/CodeFormer/weights/facelib/parsing_parsenet.pth

print("⬇️ Downloading Background Upscaler...")
!mkdir -p /content/weights/realesrgan
!wget https://github.com/sczhou/CodeFormer/releases/download/v0.1.0/RealESRGAN_x2plus.pth -O /content/weights/realesrgan/RealESRGAN_x2plus.pth

print("✅ All models downloaded manually. No more download script needed!")

⬇️ Downloading CodeFormer model...
--2026-01-04 19:14:16--  https://github.com/sczhou/CodeFormer/releases/download/v0.1.0/codeformer.pth
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/505667511/a1f9f85b-f048-428c-b18b-b79b2665a325?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-01-04T19%3A59%3A16Z&rscd=attachment%3B+filename%3Dcodeformer.pth&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-01-04T18%3A58%3A43Z&ske=2026-01-04T19%3A59%3A16Z&sks=b&skv=2018-11-09&sig=chYg72m5qevqxpdw2NVD5b443QMaUkS2bVqampZqSRc%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc2NzU1NzY1NiwibmJmIjoxNzY3NTU0MDU2LCJwYXRoIjo

In [ ]:
# Cell: Run Path B Inference (Final Texture Pass)
%cd /content/CodeFormer

# We use -w 0.5 to balance AI sharpness with the original baby's features
# This prevents the "uncanny valley" look
print("🚀 Running CodeFormer... Adding textures to your 12 images.")
!PYTHONPATH=. python inference_codeformer.py -w 0.5 --has_aligned --input_path /content/inputs --bg_upsampler realesrgan --face_upsample

# 2. Organize Results
import os
import shutil

source_dir = "results/final_results"
dest_dir = "/content/results/path_b_codeformer"
os.makedirs(dest_dir, exist_ok=True)

if os.path.exists(source_dir):
    files = os.listdir(source_dir)
    print(f"📦 Moving {len(files)} sharpened images to Path B...")
    for f in files:
        shutil.move(os.path.join(source_dir, f), os.path.join(dest_dir, f))
    print(f"✅ Path B Complete! Results are in: {dest_dir}")
else:
    print("⚠️ Hmm, I don't see the results. Check the log above for any CUDA or OOM errors.")

/content/CodeFormer
🚀 Running CodeFormer... Adding textures to your 12 images.
Background upsampling: True, Face upsampling: True
[1/12] Processing: 00000.png
[2/12] Processing: 00001.png
[3/12] Processing: 00002.png
[4/12] Processing: 00003.png
[5/12] Processing: 00004.png
[6/12] Processing: 00005.png
[7/12] Processing: 00006.png
[8/12] Processing: 00007.png
[9/12] Processing: 00008.png
[10/12] Processing: 00009.png
[11/12] Processing: 00010.png
[12/12] Processing: 00011.png

All results are saved in results/inputs_0.5
⚠️ Hmm, I don't see the results. Check the log above for any CUDA or OOM errors.


In [ ]:
!find /content/CodeFormer/results -name "*.png"

/content/CodeFormer/results/inputs_0.5/restored_faces/00002.png
/content/CodeFormer/results/inputs_0.5/restored_faces/00005.png
/content/CodeFormer/results/inputs_0.5/restored_faces/00006.png
/content/CodeFormer/results/inputs_0.5/restored_faces/00001.png
/content/CodeFormer/results/inputs_0.5/restored_faces/00003.png
/content/CodeFormer/results/inputs_0.5/restored_faces/00004.png
/content/CodeFormer/results/inputs_0.5/restored_faces/00009.png
/content/CodeFormer/results/inputs_0.5/restored_faces/00000.png
/content/CodeFormer/results/inputs_0.5/restored_faces/00010.png
/content/CodeFormer/results/inputs_0.5/restored_faces/00007.png
/content/CodeFormer/results/inputs_0.5/restored_faces/00011.png
/content/CodeFormer/results/inputs_0.5/restored_faces/00008.png


In [ ]:
# Cell: Prepare Textures for Fusion
import os
import shutil

# The path identified by your 'find' command
source_dir = "/content/CodeFormer/results/inputs_0.5/restored_faces"
dest_dir = "/content/results/path_b_codeformer"
os.makedirs(dest_dir, exist_ok=True)

files = os.listdir(source_dir)
print(f"📦 Found {len(files)} sharp faces. Moving to Path B...")

for f in files:
    # CodeFormer sometimes adds suffixes like '_00' or '_0.5'
    # We strip them to match '00000.png', '00001.png', etc.
    clean_name = f.split('_')[0]
    if not clean_name.endswith('.png'):
        clean_name += '.png'

    shutil.copy(os.path.join(source_dir, f), os.path.join(dest_dir, clean_name))
    print(f"   ✅ Prepared: {clean_name}")

print(f"\n✨ Path B is ready. Folder: {dest_dir}")

📦 Found 12 sharp faces. Moving to Path B...
   ✅ Prepared: 00002.png
   ✅ Prepared: 00005.png
   ✅ Prepared: 00006.png
   ✅ Prepared: 00001.png
   ✅ Prepared: 00003.png
   ✅ Prepared: 00004.png
   ✅ Prepared: 00009.png
   ✅ Prepared: 00000.png
   ✅ Prepared: 00010.png
   ✅ Prepared: 00007.png
   ✅ Prepared: 00011.png
   ✅ Prepared: 00008.png

✨ Path B is ready. Folder: /content/results/path_b_codeformer


In [ ]:
# Cell: Texture-Boosted Fusion
import cv2
import numpy as np
import glob
import os

path_a_dir = "/content/results/path_a_hat"
path_b_dir = "/content/results/path_b_codeformer"
output_dir = "/content/results/FINAL_GRANDMASTER_BOOSTED"
os.makedirs(output_dir, exist_ok=True)

# CONFIGURATION FOR PERCEPTUAL TRACK
TEXTURE_BOOST = 1.8  # Increase this (1.5 - 2.5) to lower LPIPS
BLUR_SIZE = (5, 5)   # Smaller kernel = sharper texture extraction

images_a = sorted(glob.glob(os.path.join(path_a_dir, "*.png")))

print(f"🚀 Boosting textures for {len(images_a)} images...")

for img_a_path in images_a:
    name = os.path.basename(img_a_path)
    img_b_path = os.path.join(path_b_dir, name)

    if os.path.exists(img_b_path):
        img_a = cv2.imread(img_a_path).astype(np.float32)
        img_b = cv2.imread(img_b_path).astype(np.float32)

        if img_a.shape != img_b.shape:
            img_b = cv2.resize(img_b, (img_a.shape[1], img_a.shape[0]), interpolation=cv2.INTER_LANCZOS4)

        # Extraction of High-Frequency Detail
        low_freq_b = cv2.GaussianBlur(img_b, BLUR_SIZE, 0)
        high_freq_detail = img_b - low_freq_b

        # Injection with Boost Factor
        # Result = Base Structure + (Detail * Boost)
        fusion = img_a + (high_freq_detail * TEXTURE_BOOST)
        fusion = np.clip(fusion, 0, 255).astype(np.uint8)

        cv2.imwrite(os.path.join(output_dir, name), fusion)
        print(f"   🔥 Boosted: {name}")

print(f"\n✅ Boosted results saved to: {output_dir}")

🚀 Boosting textures for 12 images...
   🔥 Boosted: 00000.png
   🔥 Boosted: 00001.png
   🔥 Boosted: 00002.png
   🔥 Boosted: 00003.png
   🔥 Boosted: 00004.png
   🔥 Boosted: 00005.png
   🔥 Boosted: 00006.png
   🔥 Boosted: 00007.png
   🔥 Boosted: 00008.png
   🔥 Boosted: 00009.png
   🔥 Boosted: 00010.png
   🔥 Boosted: 00011.png

✅ Boosted results saved to: /content/results/FINAL_GRANDMASTER_BOOSTED


In [ ]:
!pip install lpips

In [ ]:
import cv2
import numpy as np
import torch
import lpips
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
import glob
import os

# Initialize LPIPS (VGG based)
loss_fn_vgg = lpips.LPIPS(net='vgg').to('cuda')

def calculate_metrics(img1_path, img2_path):
    img1 = cv2.imread(img1_path)
    img2 = cv2.imread(img2_path)

    # Ensure same size for PSNR/SSIM
    img1 = cv2.resize(img1, (img2.shape[1], img2.shape[0]))

    # 1. PSNR
    p = psnr(img1, img2)

    # 2. SSIM
    s = ssim(img1, img2, channel_axis=2)

    # 3. LPIPS
    t1 = lpips.im2tensor(img1).to('cuda')
    t2 = lpips.im2tensor(img2).to('cuda')
    l = loss_fn_vgg(t1, t2).item()

    return p, s, l

# Paths
input_dir = "/content/inputs"
output_dir = "/content/results/FINAL_GRANDMASTER"
images = sorted(glob.glob(os.path.join(input_dir, "*.png")))

avg_psnr, avg_ssim, avg_lpips = [], [], []

print(f"🧪 Benchmarking {len(images)} images against NTIRE standards...\n")

for img_path in images:
    name = os.path.basename(img_path)
    out_path = os.path.join(output_dir, name)

    if os.path.exists(out_path):
        p, s, l = calculate_metrics(img_path, out_path)
        avg_psnr.append(p)
        avg_ssim.append(s)
        avg_lpips.append(l)
        print(f"{name} -> PSNR: {p:.2f}dB | SSIM: {s:.4f} | LPIPS: {l:.4f}")

print(f"\n--- 🏁 FINAL AVERAGES ---")
print(f"⭐ PSNR: {np.mean(avg_psnr):.2f} dB (Target > 28 for Fidelity)")
print(f"⭐ SSIM: {np.mean(avg_ssim):.4f} (Target > 0.85 for Structure)")
print(f"⭐ LPIPS: {np.mean(avg_lpips):.4f} (Target < 0.2 for Perceptual)")

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/vgg.pth
🧪 Benchmarking 12 images against NTIRE standards...

00000.png -> PSNR: 44.95dB | SSIM: 0.9839 | LPIPS: 0.2320
00001.png -> PSNR: 42.37dB | SSIM: 0.9745 | LPIPS: 0.2884
00002.png -> PSNR: 45.16dB | SSIM: 0.9864 | LPIPS: 0.2609
00003.png -> PSNR: 45.93dB | SSIM: 0.9854 | LPIPS: 0.2771
00004.png -> PSNR: 40.99dB | SSIM: 0.9762 | LPIPS: 0.2334
00005.png -> PSNR: 44.99dB | SSIM: 0.9816 | LPIPS: 0.2860
00006.png -> PSNR: 43.87dB | SSIM: 0.9794 | LPIPS: 0.2384
00007.png -> PSNR: 43.07dB | SSIM: 0.9726 | LPIPS: 0.2983
00008.png -> PSNR: 45.35dB | SSIM: 0.9826 | LPIPS: 0.2705
00009.png -> PSNR: 42.12dB | SSIM: 0.9749 | LPIPS: 0.2633
00010.png -> PSNR: 40.69dB | SSIM: 0.9788 | LPIPS: 0.1872
00011.png -> PSNR: 41.20dB | SSIM: 0.9821 | LPIPS: 0.2083

--- 🏁 FINAL AVERAGES ---
⭐ PSNR: 43.39 dB (Target > 28 for Fidelity)
⭐ SSIM: 0.9799 (Target 